# Fine-Tuning with LoRA on Apple Silicon

## What you'll learn

- Load the 4-bit `mlx-community/Llama-3.2-1B-Instruct-4bit` checkpoint.
- Train LoRA adapters while keeping the quantized base weights frozen (QLoRA).
- Use tokenizer chat templates, prompt-loss masking, and gradient checkpointing.
- Compare base and adapted models on a small intent-label task.

This teaching dataset contains 800 training and 200 validation records generated
from a small template pool. Source prompts are split before sampling repeated
records, so validation prompts do not appear in training. Targets are exactly
`greeting`, `question`, or `command`; record count is not unique-prompt count.

Memory, runtime, and quality depend on hardware, sequence length, batch size,
and data. This notebook makes no fixed memory or quality guarantee. Full LoRA
training was not executed during the October 2026 validation pass; imports,
notebook syntax, and CLI flags were checked.


In [ ]:
# Install the pinned environment from the repository root with `make setup`.
# Restart the notebook kernel after changing dependencies.


In [ ]:
import sys
from mlx_lm import load, generate
from mlx_nlp_utils import print_device_info

print_device_info()
from mlx_lm.sample_utils import make_sampler
sampler = make_sampler(temp=0.0)


## 1. Load the Pretrained Model

The code loads `mlx-community/Llama-3.2-1B-Instruct-4bit` through MLX-LM.
The first run downloads its weights and tokenizer into the Hugging Face cache.
Use the pinned project environment from `make setup`, then restart the kernel.


In [ ]:
# Load model and tokenizer
# Using Llama-3.2-1B-Instruct (a compact, established model for this exercise)
model_name = "mlx-community/Llama-3.2-1B-Instruct-4bit"
print(f"Loading {model_name}...")

model, tokenizer = load(model_name)

print("✅ Model loaded!")

## 2. Test Base Model

Let's see how it performs *before* fine-tuning.

In [ ]:
# Use the tokenizer's chat template (Modern Best Practice)
messages = [
    {"role": "user", "content": "What is the capital of France?"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"Formatted Prompt:\n{prompt}\n{'='*20}")

response = generate(model, tokenizer, prompt=prompt, max_tokens=100, sampler=sampler, verbose=True)

## 3. Prepare Label-Only Chat Data

Run `make setup-samples` from the repository root to regenerate `data/train.jsonl`
and `data/valid.jsonl`. Each JSONL record contains system, user, and assistant
messages. The assistant response is the intent label. Older conversational
adapters should be retrained for this task.


In [ ]:
import json
from pathlib import Path

# 1. Use the pre-generated LoRA chat data
# Generated by: python scripts/generate_synthetic_data.py
train_path = Path('../data/train.jsonl')
valid_path = Path('../data/valid.jsonl')

if not train_path.exists() or not valid_path.exists():
    print("⚠️ LoRA data not found. Please run: python ../scripts/generate_synthetic_data.py")
else:
    with open(train_path) as f:
        train_data = [json.loads(line) for line in f]
    with open(valid_path) as f:
        valid_data = [json.loads(line) for line in f]

    print(f"Loaded {len(train_data)} train / {len(valid_data)} valid chat records")
    print("Format: OpenAI-style {messages: [system, user, assistant]}")
    print("Sample:")
    print(json.dumps(train_data[0], indent=2))
    print("\n✅ Data ready for MLX LoRA!")

## 4. Run Fine-Tuning

The training cell invokes `sys.executable -m mlx_lm.lora` so it uses the notebook
kernel's environment, and `check=True` surfaces training failure. It runs 200
iterations with batch size 2, a 256-token sequence limit, gradient checkpointing,
and `--mask-prompt` to train on the assistant answer rather than the prompt.
Reduce batch/sequence size if necessary for your device.


In [ ]:
# Training Configuration
# We use the CLI tool provided by mlx-lm
# --batch-size 2: A small starting point; reduce further if memory is tight
# --num-layers 16: Target more layers for better quality
# --iters 200: Appropriate for ~800 training examples

print("📋 Training Command (for reference):")
print("="*60)

cmd = r"""python -m mlx_lm.lora \
    --model mlx-community/Llama-3.2-1B-Instruct-4bit \
    --train \
    --data ../data \
    --batch-size 2 \
    --num-layers 16 \
    --grad-checkpoint \
    --mask-prompt \
    --max-seq-length 256 \
    --iters 200 \
    --learning-rate 1e-4 \
    --adapter-path ./adapters"""

print(cmd)
print("="*60)
print("\n💡 Recommendation: Run the next cell to train in the notebook!")
print("   (It handles paths automatically)")

In [ ]:
# Run LoRA Training (5-10 minutes on M1/M2/M3)
import os
import sys
from pathlib import Path

# Disable tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("🚀 Starting LoRA Fine-Tuning...")
print("="*60)

# 1. Setup Paths
current_dir = Path.cwd()

# Find project root and working directory
if (current_dir / "notebooks").exists():
    project_root = current_dir
    working_dir = current_dir / "notebooks"
elif current_dir.name == "notebooks":
    project_root = current_dir.parent
    working_dir = current_dir
else:
    project_root = current_dir
    working_dir = current_dir

data_dir = project_root / "data"
adapter_dir = working_dir / "adapters"
adapter_dir.mkdir(exist_ok=True)

print(f"📍 Project Root: {project_root}")
print(f"📂 Data Dir: {data_dir}")
print(f"💾 Adapter Dir: {adapter_dir}")

# 2. Verify Data Exists
if not (data_dir / "train.jsonl").exists():
    print("\n❌ ERROR: 'train.jsonl' not found.")
    print("   Please run: python scripts/generate_synthetic_data.py")
else:
    print("\n✅ Data found. Starting training...")
    print("="*60)
    
    # Use the active kernel's Python and fail if training exits unsuccessfully.
    import subprocess
    subprocess.run([
        sys.executable, "-m", "mlx_lm.lora", "--model", model_name,
        "--train", "--data", str(data_dir), "--batch-size", "2",
        "--num-layers", "16", "--iters", "200", "--learning-rate", "1e-4",
        "--grad-checkpoint", "--mask-prompt", "--max-seq-length", "256",
        "--adapter-path", str(adapter_dir),
    ], check=True)
    print(f"Training complete. Adapters saved to: {adapter_dir}")


### What training does

MLX-LM loads the base model and data, adds trainable adapters to the selected
layers, and saves them under `./adapters`. The inference cells check for
`adapters.safetensors`; an empty directory does not count as a trained model.
Both base and adapted inference use the task's system instruction and greedy
sampling through `make_sampler(temp=0.0)`.

The small template dataset demonstrates mechanics. For an application, build
representative evaluation data and tune rank, learning rate, and training budget
against it. More iterations alone do not guarantee better held-out results.


## 5. Inference with Fine-Tuned Model

After training completes, we can load the base model + adapters and test it.

In [ ]:
# Load the fine-tuned model (base + adapters) if available
from pathlib import Path

adapter_path = Path("./adapters")

if (adapter_path / "adapters.safetensors").is_file():
    print("✅ Loading fine-tuned model (base + LoRA adapters)...")
    model_ft, tokenizer_ft = load(model_name, adapter_path=str(adapter_path))
    print("✅ Fine-tuned model loaded!")
else:
    print("⚠️  No adapters found. Will use base model only.")
    print("   (Run the training command first to create adapters)")
    model_ft, tokenizer_ft = model, tokenizer

# Side-by-side comparison
test_query = "Turn off the lights"

messages = [
    {"role": "system", "content": "Classify the user message. Reply with exactly one label: greeting, question, or command."},
    {"role": "user", "content": test_query}
]
prompt_test = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(f"\nTest Query: '{test_query}'")
print("="*60)

print("\n📦 BASE MODEL:")
response_base = generate(model, tokenizer, prompt=prompt_test, max_tokens=50, sampler=sampler, verbose=False)
print(response_base)

if (adapter_path / "adapters.safetensors").is_file():
    print("\n🎯 FINE-TUNED MODEL:")
    response_ft = generate(model_ft, tokenizer_ft, prompt=prompt_test, max_tokens=50, sampler=sampler, verbose=False)
    print(response_ft)
else:
    print("\n⚠️  Fine-tuned model not available (run training first)")

print("="*60)
print("\n💡 Expected Improvement:")
print("   - Base model: Generic response or incorrect intent")
print("   - Fine-tuned: Correctly identifies 'command'")

## 6. Compare: Base vs Fine-Tuned

Let's see the difference between the base model and our fine-tuned version.

In [ ]:
# Test the fine-tuned model on intent classification
if not (adapter_path / "adapters.safetensors").is_file():
    print("⚠️  No fine-tuned model available. Run training first.")
    print("   Using base model for testing.")

test_cases = [
    "Turn on the light",
    "What time is it",
    "Hello there",
    "Set a timer for 5 minutes",
    "How are you doing"
]

model_to_test = model_ft if (adapter_path / "adapters.safetensors").is_file() else model
tokenizer_to_test = tokenizer_ft if (adapter_path / "adapters.safetensors").is_file() else tokenizer
model_type = "Fine-Tuned" if (adapter_path / "adapters.safetensors").is_file() else "Base"

print(f"\n🧪 Testing {model_type} Model\n" + "="*50)

for test_text in test_cases:
    messages = [
        {"role": "system", "content": "Classify the user message. Reply with exactly one label: greeting, question, or command."},
        {"role": "user", "content": test_text}
    ]
    
    prompt = tokenizer_to_test.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    print(f"\n📝 Input: '{test_text}'")
    response = generate(model_to_test, tokenizer_to_test, prompt=prompt, max_tokens=50, sampler=sampler, verbose=False)
    print(f"🤖 Output: {response}")

print("\n" + "="*50)

## Further Experiments: Optimizing LoRA

Fine-tuning is an art as much as a science. Here are some experiments to deepen your understanding.

### Experiment: Finding the Optimal Rank
We used `lora_rank=8`. What happens if we change it?
- **Rank 4:** Faster, less memory, but might not learn complex patterns.
- **Rank 32:** Slower, more memory, potentially better quality.
- **Rank 64+:** Diminishing returns, high risk of overfitting.

In [ ]:
# Experiment: Find optimal LoRA rank

def test_lora_ranks():
    """Compare different LoRA ranks on same task"""
    print("Testing LoRA Ranks...")
    
    ranks = [4, 8, 16, 32]
    
    # In a real experiment, we would loop through these ranks, 
    # train a model for each, and evaluate on a validation set.
    
    for rank in ranks:
        print(f"Training with Rank {rank}...")
        # train_lora(rank=rank)
        # score = evaluate()
        # print(f"Rank {rank} Score: {score}")
        pass
        
    print("General Rule: Rank 8 or 16 is usually the sweet spot for 7B models.")

# test_lora_ranks()

## 🎉 Conclusion

You have now completed the journey from simple RNNs to fine-tuning modern LLMs on Apple Silicon!

**What's Next?**
- Build a RAG (Retrieval Augmented Generation) system.
- Deploy your fine-tuned model as an API.

## FAQ

**How is this different from full fine-tuning?**
Full fine-tuning updates the base weights. Here the quantized base is frozen and
only adapters are trained. The trainable fraction depends on the adapter rank
and target layers; measure memory and quality for your own configuration.

**Can I use my own data?**
Yes. Preserve the JSONL `messages` format and make the assistant targets match
the behavior you will evaluate. Split prompts/documents before creating repeated
or augmented examples so validation does not leak into training.

**What does 4-bit mean?**
It describes the base weight representation, not total training memory. Runtime
also needs quantization metadata, activations, adapters, and optimizer state.

For CLI configuration details, see [MLX-LM's LoRA guide](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md).
